# Gold Layer — checks (design 1.3 / 1.4 / 1.5)

Run after every Gold load. Each check is a query that must return **0**. Any failure raises, so the daily job stops before `agg_restaurant_ranking` is rebuilt.

- **Grain:** one row per natural key in every dimension, one row per grain in every fact; exactly one Unknown (−1) row per dimension.
- **Orphans:** every fact key exists in its dimension; no fact key is NULL.
- **No counters in dims:** counters belong to the snapshot facts.
- **Dense snapshots (warning):** from `DENSE_FROM` (first day of the daily refresh) every day should have a snapshot. A missed day cannot be backfilled (the YouTube API has no history), and `views_gained` after a gap still adds up correctly, so a gap is reported but does not stop the job. While the schedule is paused and the job runs by hand, gaps are expected.
- **Restaurant mentions** (once built): `SUM(mention_weight) = 1` per video, one current version per `restaurant_id`.

Warnings (printed, not failing): `view_count` dropping between two snapshots; days without a snapshot since `DENSE_FROM`.


In [0]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

DENSE_FROM = "2026-10-28"  # dense daily snapshots start here; 2026-10-27 was never captured (job skipped), earlier history is sparse


In [0]:
def table_exists(name):
    return spark.catalog.tableExists(f"{GOLD}.{name}")


checks = {
    # grain: dimensions
    "dim_date: duplicate date_key": f"SELECT COUNT(*) - COUNT(DISTINCT date_key) FROM {GOLD}.dim_date",
    "dim_video: duplicate video_id": f"SELECT COUNT(*) - COUNT(DISTINCT video_id) FROM {GOLD}.dim_video WHERE video_key <> -1",
    "dim_channel: duplicate channel_id": f"SELECT COUNT(*) - COUNT(DISTINCT channel_id) FROM {GOLD}.dim_channel WHERE channel_key <> -1",
    "dim_commenter: duplicate commenter_key": f"SELECT COUNT(*) - COUNT(DISTINCT commenter_key) FROM {GOLD}.dim_commenter",
    "dim_keyword: duplicate keyword_normalized": f"SELECT COUNT(*) - COUNT(DISTINCT keyword_normalized) FROM {GOLD}.dim_keyword WHERE keyword_key <> -1",
    # grain: facts
    "fact_video_daily_snapshot: duplicate grain": f"SELECT COUNT(*) - COUNT(DISTINCT snapshot_date_key, video_key) FROM {GOLD}.fact_video_daily_snapshot",
    "fact_channel_daily_snapshot: duplicate grain": f"SELECT COUNT(*) - COUNT(DISTINCT snapshot_date_key, channel_key) FROM {GOLD}.fact_channel_daily_snapshot",
    "fact_comment: duplicate comment_id": f"SELECT COUNT(*) - COUNT(DISTINCT comment_id) FROM {GOLD}.fact_comment",
    "fact_search_hit: duplicate grain": f"SELECT COUNT(*) - COUNT(DISTINCT fetched_at, keyword_key, video_key) FROM {GOLD}.fact_search_hit",
    # Unknown members
    **{
        f"{dim}: not exactly one Unknown row": f"SELECT ABS(COUNT_IF({key} = -1) - 1) FROM {GOLD}.{dim}"
        for dim, key in [("dim_date", "date_key"), ("dim_video", "video_key"), ("dim_channel", "channel_key"),
                         ("dim_commenter", "commenter_key"), ("dim_keyword", "keyword_key")]
    },
}

# dense snapshots: warning only (a missed day cannot be backfilled; views_gained after a gap still adds up)
missing_days_sql = {
    fact: f"""
        SELECT collect_list(CAST(d.full_date AS STRING)) FROM (
          SELECT full_date FROM {GOLD}.dim_date d
          WHERE d.full_date BETWEEN DATE'{DENSE_FROM}' AND (SELECT to_date(CAST(MAX(snapshot_date_key) AS STRING), 'yyyyMMdd') FROM {GOLD}.{fact})
            AND d.date_key NOT IN (SELECT DISTINCT snapshot_date_key FROM {GOLD}.{fact})
          ORDER BY full_date) d"""
    for fact in ("fact_video_daily_snapshot", "fact_channel_daily_snapshot")
}

# orphans and NULL keys: (fact, fact column, dimension, dimension key)
FACT_KEYS = [
    ("fact_video_daily_snapshot", "snapshot_date_key", "dim_date", "date_key"),
    ("fact_video_daily_snapshot", "video_key", "dim_video", "video_key"),
    ("fact_video_daily_snapshot", "channel_key", "dim_channel", "channel_key"),
    ("fact_channel_daily_snapshot", "snapshot_date_key", "dim_date", "date_key"),
    ("fact_channel_daily_snapshot", "channel_key", "dim_channel", "channel_key"),
    ("fact_comment", "posted_date_key", "dim_date", "date_key"),
    ("fact_comment", "video_key", "dim_video", "video_key"),
    ("fact_comment", "channel_key", "dim_channel", "channel_key"),
    ("fact_comment", "commenter_key", "dim_commenter", "commenter_key"),
    ("fact_search_hit", "fetched_date_key", "dim_date", "date_key"),
    ("fact_search_hit", "keyword_key", "dim_keyword", "keyword_key"),
    ("fact_search_hit", "video_key", "dim_video", "video_key"),
    ("fact_search_hit", "channel_key", "dim_channel", "channel_key"),
]

# restaurant tables are built later (design step 6); check them once they exist
if table_exists("dim_restaurant"):
    # closed restaurants (left the source) have no current version, so the rule is "at most one"
    checks["dim_restaurant: more than one current version per restaurant_id"] = f"""
        SELECT COUNT(*) FROM (SELECT restaurant_id FROM {GOLD}.dim_restaurant WHERE restaurant_key <> -1
                              GROUP BY restaurant_id HAVING COUNT_IF(is_current) > 1)"""
    checks["dim_restaurant: mentioned restaurant without a current version"] = f"""
        SELECT COUNT(*) FROM {GOLD}.fact_video_restaurant_mention f
        JOIN {GOLD}.dim_restaurant d ON d.restaurant_key = f.restaurant_key
        LEFT ANTI JOIN {GOLD}.dim_restaurant c ON c.restaurant_id = d.restaurant_id AND c.is_current"""
    checks["dim_restaurant: duplicate restaurant_key"] = f"SELECT COUNT(*) - COUNT(DISTINCT restaurant_key) FROM {GOLD}.dim_restaurant"
if table_exists("fact_video_restaurant_mention"):
    checks["fact_video_restaurant_mention: duplicate grain"] = f"SELECT COUNT(*) - COUNT(DISTINCT video_key, restaurant_key) FROM {GOLD}.fact_video_restaurant_mention"
    checks["fact_video_restaurant_mention: SUM(mention_weight) <> 1 per video"] = f"""
        SELECT COUNT(*) FROM (SELECT video_key FROM {GOLD}.fact_video_restaurant_mention
                              GROUP BY video_key HAVING ABS(SUM(mention_weight) - 1) > 1e-9)"""
    FACT_KEYS += [("fact_video_restaurant_mention", "video_key", "dim_video", "video_key"),
                  ("fact_video_restaurant_mention", "restaurant_key", "dim_restaurant", "restaurant_key")]

for fact, col, dim, key in FACT_KEYS:
    checks[f"{fact}.{col}: NULL"] = f"SELECT COUNT(*) FROM {GOLD}.{fact} WHERE {col} IS NULL"
    checks[f"{fact}.{col}: orphan (not in {dim})"] = f"""
        SELECT COUNT(*) FROM {GOLD}.{fact} f LEFT ANTI JOIN {GOLD}.{dim} d ON d.{key} = f.{col}"""

# no counters in dimensions
for dim in ["dim_video", "dim_channel"]:
    counters = [c for c in spark.table(f"{GOLD}.{dim}").columns if c.endswith("_count")]
    checks[f"{dim}: holds counters {counters}"] = f"SELECT {len(counters)}"


In [0]:
# --- Run ---
failures = []
for name, sql in checks.items():
    bad = spark.sql(sql).collect()[0][0] or 0
    print(("❌" if bad else "✅") + f" {name}: {bad}")
    if bad:
        failures.append(f"{name}: {bad}")

drops = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.fact_video_daily_snapshot WHERE views_gained < 0").collect()[0][0]
print(("⚠️" if drops else "✅") + f" fact_video_daily_snapshot: rows where view_count dropped: {drops} (warning only)")
for fact, sql in missing_days_sql.items():
    missing = spark.sql(sql).collect()[0][0] or []
    print(("⚠️" if missing else "✅") + f" {fact}: days without a snapshot since {DENSE_FROM}: {missing} (warning only)")

if failures:
    raise AssertionError(f"{len(failures)} Gold check(s) failed:\n" + "\n".join(failures))
print(f"\n--- All {len(checks)} Gold checks passed ---")
